# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/devotedhruv/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [3]:
import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfFileSystem
from google.colab import userdata

# Authenticate
hf_token = userdata.get('HF_TOKEN')

# 1. Let's look inside the repository to find the exact folder structure
fs = HfFileSystem(token=hf_token)
print("Files in the root directory:")
root_files = fs.ls("datasets/FlyRank/internship-warehouse", detail=False)
for f in root_files:
    print(f)

# 2. Once you see the correct folder name in the print output above,
# you can update the data_files path.
# It might just be "month=2026-03/*.parquet" or something similar.

# dataset = load_dataset(
#     "FlyRank/internship-warehouse",
#     data_files="CORRECT_PATH_FROM_ABOVE/*.parquet",
#     token=hf_token,
#     split="train"
# )
# df = dataset.to_pandas()

# # Verify the grain
# duplicate_count = df.duplicated(subset=['date', 'query', 'url']).sum()
# print(f"\nGrain verification: {duplicate_count} duplicate rows found (Expected: 0).")

# # Verify the time window
# min_date = df['date'].min()
# max_date = df['date'].max()
# print(f"Time window verification: {min_date} to {max_date}")
# print(f"Total rows in this slice: {len(df)}")

Files in the root directory:
datasets/FlyRank/internship-warehouse/fact_content_daily_performance
datasets/FlyRank/internship-warehouse/.gitattributes
datasets/FlyRank/internship-warehouse/README.md
datasets/FlyRank/internship-warehouse/dim_clients.parquet
datasets/FlyRank/internship-warehouse/dim_content.parquet
datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet
datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [5]:
import pandas as pd
from datasets import load_dataset
from google.colab import userdata

# 1. LOAD THE DATASET (This defines 'df')
try:
    hf_token = userdata.get('HF_TOKEN')

    # Loading the March 2026 partition.
    # (If this throws a DataFilesNotFoundError, change the data_files path to match the exact folder name in the Hugging Face repo)
    dataset = load_dataset(
        "FlyRank/internship-warehouse",
        data_files="month=2026-03/*.parquet",
        token=hf_token,
        split="train"
    )
    df = dataset.to_pandas()
    print("✅ Dataset loaded successfully!\n")

except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    print("If you see a DataFilesNotFoundError, you need to check the Hugging Face repo to find the exact folder path for the March 2026 data.")

# 2. APPLY EXCLUSIONS & CATEGORIZE FIELDS (Only runs if df was loaded)
if 'df' in locals():
    # 1. Exclude rows with fewer than 5 impressions to remove long-tail noise
    df_filtered = df[df['impressions'] >= 5].copy()

    # 2. Exclude CTR to prevent target leakage (if the column exists)
    if 'ctr' in df_filtered.columns:
        df_filtered = df_filtered.drop(columns=['ctr'])

    # Create the binary label
    df_filtered['has_click'] = (df_filtered['clicks'] > 0).astype(int)

    # Define the field buckets programmatically
    context_cols = ['date']
    feature_cols = ['query', 'url', 'device', 'impressions', 'position']
    label_col = ['has_click']

    # Output the results of the sorting and exclusion
    print(f"Rows before exclusion: {len(df)}")
    print(f"Rows after excluding < 5 impressions: {len(df_filtered)}")
    print(f"Context Fields: {context_cols}")
    print(f"Feature Fields: {feature_cols}")
    print(f"Label Field: {label_col} (Derived from 'clicks')")

❌ Error loading dataset: No (supported) data files found in FlyRank/internship-warehouse
If you see a DataFilesNotFoundError, you need to check the Hugging Face repo to find the exact folder path for the March 2026 data.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
import pandas as pd
from datasets import load_dataset
from google.colab import userdata

# 1. LOAD THE DATASET
hf_token = userdata.get('HF_TOKEN')

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/*.parquet",
    token=hf_token,
    split="train"
)
df = dataset.to_pandas()

# Display schema details to verify exact column names
print("=== DATASET SCHEMA ===")
print("Columns:", df.columns.tolist())
print("\nFirst 2 rows:")
display(df.head(2))
print("=" * 30 + "\n")

# Detect columns dynamically or set fallbacks
date_col = next((c for c in df.columns if 'date' in c.lower() or c in ['dt', 'day']), df.columns[0])
query_col = next((c for c in df.columns if 'query' in c.lower() or 'term' in c.lower() or 'keyword' in c.lower()), None)
url_col = next((c for c in df.columns if 'url' in c.lower() or 'page' in c.lower() or 'path' in c.lower()), None)

# Determine available ID columns for grain check
grain_cols = [c for c in [date_col, query_col, url_col] if c is not None]
if 'device' in df.columns:
    grain_cols.append('device')

# 2. QUERY 1: Verify the Grain
print("--- Query 1: Grain Verification ---")
if grain_cols:
    duplicates = df.duplicated(subset=grain_cols).sum()
    print(f"Grain columns checked: {grain_cols}")
    print(f"Duplicate rows found: {duplicates} (Expected: 0)")
else:
    print("Could not infer grain columns automatically. Columns found:", df.columns.tolist())

# 3. QUERY 2: Verify Counts and Date Span
print("\n--- Query 2: Slice Counts & Date Span ---")
total_rows = len(df)
min_date = df[date_col].min()
max_date = df[date_col].max()
print(f"Total row count: {total_rows}")
print(f"Date span: {min_date} to {max_date}")

# 4. QUERY 3: Verify Availability (IS TRUE)
print("\n--- Query 3: Availability (IS TRUE Check) ---")
# If an availability boolean flag exists (e.g., is_available, is_active, is_indexed), use it;
# otherwise create a completeness boolean mask across primary metric/dimension columns.
if 'is_available' in df.columns:
    surviving_df = df[df['is_available'] == True]
else:
    # Filter where non-null across key columns
    df['is_complete'] = df[grain_cols].notna().all(axis=1)
    surviving_df = df[df['is_complete'] == True]

surviving_count = len(surviving_df)
print(f"Surviving rows (IS TRUE): {surviving_count} out of {total_rows}")

=== DATASET SCHEMA ===
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']

First 2 rows:


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



--- Query 1: Grain Verification ---
Grain columns checked: ['report_date', 'ga4_pageviews']
Duplicate rows found: 9839363 (Expected: 0)

--- Query 2: Slice Counts & Date Span ---
Total row count: 9841378
Date span: 2026-03-01 to 2026-03-31

--- Query 3: Availability (IS TRUE Check) ---
Surviving rows (IS TRUE): 6822637 out of 9841378


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Demonstrate the "Unbalanced history" limitation:
# Let's count how many URLs have a full month of data vs. those that appear on just 1 or 2 days.
# This proves the model will have vastly different amounts of historical context per URL.

if url_col in df.columns and date_col in df.columns:
    # Count unique days each URL appears in the dataset
    url_history_counts = df.groupby(url_col)[date_col].nunique()

    total_unique_urls = len(url_history_counts)
    single_day_urls = (url_history_counts == 1).sum()
    full_month_urls = (url_history_counts >= 28).sum()

    print(f"--- Unbalanced History Check ---")
    print(f"Total unique URLs in this month: {total_unique_urls:,}")
    print(f"URLs with exactly 1 day of history: {single_day_urls:,} ({(single_day_urls/total_unique_urls)*100:.1f}%)")
    print(f"URLs with 28+ days of history: {full_month_urls:,} ({(full_month_urls/total_unique_urls)*100:.1f}%)")
    print("\nImpact: A feature like 'historical_click_rate' will be highly unreliable for the new/rare URLs.")
else:
    print("Required columns for historical check are missing from the dataframe schema.")

--- Unbalanced History Check ---
Total unique URLs in this month: 234
URLs with exactly 1 day of history: 73 (31.2%)
URLs with 28+ days of history: 29 (12.4%)

Impact: A feature like 'historical_click_rate' will be highly unreliable for the new/rare URLs.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.